# ML Lab Practical 2: Email Spam Classification

**Algorithms:** K-Nearest Neighbors (KNN) and Support Vector Machine (SVM)

**Aim:** To classify emails as spam or non-spam using TF-IDF features, train KNN and SVM classifiers, and compare their performance.

**Dataset:** `emails.csv`  
**Required columns:**
- `text`: email message
- `spam`: class label (`0` = not spam, `1` = spam)

**Workflow:** Load data → Clean data → Split data → TF-IDF → Train KNN and SVM → Evaluate → Compare.

> Keep `emails.csv` in the same folder as this notebook, or update the file path in the dataset-loading cell.


## 1. Import Required Libraries

We use:
- **Pandas** to load and manipulate the dataset.
- **train_test_split** to divide data into training and testing sets.
- **TfidfVectorizer** to convert email text into numerical features.
- **KNeighborsClassifier** and **SVC** to build the two classifiers.
- **Evaluation metrics** to measure model performance.


In [1]:
# 1. Import Libraries

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Display all columns clearly when inspecting the dataset
pd.set_option("display.max_columns", None)


## 2. Load and Inspect the Dataset

Read `emails.csv` into a Pandas DataFrame and inspect its size, first few rows, column names, missing values, and class labels.

The dataset must contain the columns `text` and `spam`.


In [2]:
# 2. Load and Inspect Dataset

data = pd.read_csv("emails.csv")

print("Dataset Shape:", data.shape)
print("\nFirst Five Records:")
print(data.head())

print("\nColumns:")
print(data.columns.tolist())

# Check that the required columns are available
required_columns = {"text", "spam"}
missing_columns = required_columns - set(data.columns)

if missing_columns:
    raise ValueError(f"Dataset is missing required column(s): {sorted(missing_columns)}")

print("\nMissing Values:")
print(data[["text", "spam"]].isnull().sum())

print("\nUnique spam labels before cleaning:")
print(data["spam"].unique())


FileNotFoundError: [Errno 2] No such file or directory: 'emails.csv'

## 3. Clean the Dataset

Cleaning steps:
1. Remove duplicate rows.
2. Remove rows with missing email text or labels.
3. Convert labels to numeric values and remove labels that cannot be converted.
4. Remove blank email messages.
5. Check that the target labels are exactly `0` and `1`.

This practical uses `0` for not spam and `1` for spam.


In [ ]:
# 3. Clean Data

# Remove duplicate records
data = data.drop_duplicates()

# Remove rows where email text or spam label is missing
data = data.dropna(subset=["text", "spam"]).copy()

# Convert email messages to strings and remove leading/trailing spaces
data["text"] = data["text"].astype(str).str.strip()

# Remove empty email messages
data = data[data["text"] != ""].copy()

# Convert target labels to numeric values; invalid labels become NaN
data["spam"] = pd.to_numeric(data["spam"], errors="coerce")
data = data.dropna(subset=["spam"]).copy()

# Ensure that the labels are binary: 0 = not spam, 1 = spam
data["spam"] = data["spam"].astype(int)
valid_labels = {0, 1}

if not set(data["spam"].unique()).issubset(valid_labels):
    raise ValueError("The 'spam' column must contain only 0 (not spam) and 1 (spam).")

print("Cleaned Dataset Shape:", data.shape)
print("\nClass Distribution:")
print(data["spam"].value_counts().sort_index())

# Stratified splitting requires at least two records in each class
if data["spam"].nunique() != 2 or data["spam"].value_counts().min() < 2:
    raise ValueError("The dataset must contain at least two records in each class (0 and 1).")


## 4. Select Features and Split the Dataset

- `X` contains the input email messages.
- `y` contains the target class labels.
- 80% of the records are used for training and 20% for testing.
- `random_state=42` makes the split reproducible.
- `stratify=y` preserves approximately the same class proportions in both sets.


In [ ]:
# 4. Select Features and Split Dataset

X = data["text"]
y = data["spam"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("\nTraining class distribution:")
print(y_train.value_counts().sort_index())
print("\nTesting class distribution:")
print(y_test.value_counts().sort_index())


## 5. Convert Email Text into TF-IDF Features

Machine learning algorithms require numerical input. `TfidfVectorizer` converts email messages into numerical feature vectors.

- `stop_words="english"` removes common English words such as “the”, “is”, and “and”.
- `max_features=5000` keeps at most 5,000 terms.
- `fit_transform()` learns the vocabulary and transforms the training messages.
- `transform()` applies the same learned vocabulary to the test messages.

**Important:** We fit the vectorizer only on the training data to avoid information leakage from the test set.


In [ ]:
# 5. TF-IDF Feature Extraction

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

# Learn vocabulary from training data and convert training text to TF-IDF
X_train_tfidf = vectorizer.fit_transform(X_train)

# Convert test text using the vocabulary learned from training data
X_test_tfidf = vectorizer.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)
print("Number of TF-IDF features:", len(vectorizer.get_feature_names_out()))


## 6. Train and Evaluate K-Nearest Neighbors (KNN)

KNN is a supervised classification algorithm. It predicts the class of a new example by looking at the classes of its nearest training examples.

Here, `n_neighbors=5` means the five nearest training examples are considered when making a prediction.


In [ ]:
# 6. Train and Evaluate KNN

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_tfidf, y_train)

# Predict labels for the test data
y_pred_knn = knn.predict(X_test_tfidf)

# Calculate evaluation metrics
knn_accuracy = accuracy_score(y_test, y_pred_knn)
knn_precision = precision_score(y_test, y_pred_knn, zero_division=0)
knn_recall = recall_score(y_test, y_pred_knn, zero_division=0)
knn_f1 = f1_score(y_test, y_pred_knn, zero_division=0)

print("----- KNN PERFORMANCE -----")
print("Accuracy :", knn_accuracy)
print("Precision:", knn_precision)
print("Recall   :", knn_recall)
print("F1 Score :", knn_f1)

print("\nClassification Report - KNN")
print(classification_report(
    y_test,
    y_pred_knn,
    labels=[0, 1],
    target_names=["Not Spam", "Spam"],
    zero_division=0
))

print("KNN Confusion Matrix (rows = actual, columns = predicted):")
print(confusion_matrix(y_test, y_pred_knn, labels=[0, 1]))


## 7. Train and Evaluate Support Vector Machine (SVM)

SVM is a supervised learning algorithm that finds a decision boundary (hyperplane) to separate classes. A linear SVM is suitable for many high-dimensional text-classification tasks.

`kernel="linear"` specifies a linear decision boundary.


In [ ]:
# 7. Train and Evaluate SVM

svm = SVC(kernel="linear")
svm.fit(X_train_tfidf, y_train)

# Predict labels for the test data
y_pred_svm = svm.predict(X_test_tfidf)

# Calculate evaluation metrics
svm_accuracy = accuracy_score(y_test, y_pred_svm)
svm_precision = precision_score(y_test, y_pred_svm, zero_division=0)
svm_recall = recall_score(y_test, y_pred_svm, zero_division=0)
svm_f1 = f1_score(y_test, y_pred_svm, zero_division=0)

print("----- SVM PERFORMANCE -----")
print("Accuracy :", svm_accuracy)
print("Precision:", svm_precision)
print("Recall   :", svm_recall)
print("F1 Score :", svm_f1)

print("\nClassification Report - SVM")
print(classification_report(
    y_test,
    y_pred_svm,
    labels=[0, 1],
    target_names=["Not Spam", "Spam"],
    zero_division=0
))

print("SVM Confusion Matrix (rows = actual, columns = predicted):")
print(confusion_matrix(y_test, y_pred_svm, labels=[0, 1]))


## 8. Compare KNN and SVM

The models are compared using accuracy, precision, recall, and F1-score.

- **Accuracy:** Overall proportion of correct predictions.
- **Precision:** Of the emails predicted as spam, how many were actually spam?
- **Recall:** Of the actual spam emails, how many were identified?
- **F1-score:** Harmonic mean of precision and recall.

For spam detection, do not judge a model using accuracy alone. Precision and recall are also important.


In [ ]:
# 8. Model Comparison

results = pd.DataFrame({
    "Model": ["KNN", "SVM"],
    "Accuracy": [knn_accuracy, svm_accuracy],
    "Precision": [knn_precision, svm_precision],
    "Recall": [knn_recall, svm_recall],
    "F1 Score": [knn_f1, svm_f1]
})

print("----- MODEL COMPARISON -----")
print(results.to_string(index=False))

# Compare models using F1-score, which balances precision and recall
if svm_f1 > knn_f1:
    print("\nSVM performs better than KNN based on F1-score.")
elif knn_f1 > svm_f1:
    print("\nKNN performs better than SVM based on F1-score.")
else:
    print("\nBoth models have the same F1-score.")

print("\nNote: The actual best model depends on the results obtained from your dataset.")


## 9. Conclusion

In this practical, an email spam classification system was developed using two supervised machine learning algorithms: K-Nearest Neighbors (KNN) and Support Vector Machine (SVM).

Email messages were cleaned and converted into numerical features using TF-IDF. Both models were trained on the training data and evaluated on unseen test data using accuracy, precision, recall, F1-score, classification reports, and confusion matrices.

The model with the more suitable evaluation results can be selected for spam detection. The final conclusion about which model performs better must be based on the actual output obtained after running the notebook.

---

## Important Execution Notes

1. Install required libraries if they are not already installed:

   `pip install pandas scikit-learn jupyter`

2. Keep `emails.csv` in the same folder as this notebook.
3. Confirm that the CSV contains `text` and `spam` columns.
4. Confirm that `spam` uses `0` for not spam and `1` for spam.
5. Run the notebook from the first cell to the last cell.
6. Your numerical results may differ depending on the dataset.
